In [6]:
!pip install -U "transformers>=4.37" "datasets>=2.14" accelerate evaluate seqeval

In [7]:
#@title 1) Install libs & mount Drive
!pip install -q transformers datasets seqeval wandb accelerate
!pip install -q "git+https://github.com/huggingface/transformers.git@main"
from google.colab import drive
drive.mount('/content/drive')
# Optional: login to wandb if you use it
# !wandb login YOUR_WANDB_API_KEY

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
#@title Fixed Cell 2 — robust load & auto-clean for /content/NERfinance.csv
import os, shutil, traceback
import pandas as pd
import csv

# Path where you uploaded the CSV in Colab
DATA_PATH = '/content/NERfinance.csv'
print("Attempting to load dataset from:", DATA_PATH)
df = None
last_error = None

def try_read(path, **kwargs):
    try:
        return pd.read_csv(path, **kwargs)
    except Exception as e:
        raise

# Attempt strategies in order
strategies = [
    {"desc":"default C engine", "kwargs":{}},
    {"desc":"python engine (more permissive)", "kwargs":{"engine":"python"}},
    {"desc":"python engine, on_bad_lines='warn'", "kwargs":{"engine":"python", "on_bad_lines":"warn"}},
    {"desc":"python engine, sep='\\t' (TSV)", "kwargs":{"engine":"python", "sep":"\t", "on_bad_lines":"warn"}},
    {"desc":"no quoting (QUOTE_NONE) with escapechar='\\\\'", "kwargs":{"engine":"python", "quoting":csv.QUOTE_NONE, "escapechar":"\\", "on_bad_lines":"warn"}},
]

used_path = None
for s in strategies:
    try:
        print(f"Trying: {s['desc']}")
        df = try_read(DATA_PATH, **s["kwargs"])
        print("Success with strategy:", s["desc"])
        used_path = DATA_PATH
        break
    except Exception as e:
        last_error = e
        print(f"Strategy failed ({s['desc']}):", type(e).__name__, str(e).splitlines()[0])

# If still failed, perform balanced-quote cleanup and retry
if df is None:
    print("\nAll quick strategies failed. Attempting balanced-quote cleanup to produce a cleaned CSV...")
    cleaned_path = DATA_PATH.replace('.csv', '.cleaned.csv')
    try:
        written = 0
        with open(DATA_PATH, 'r', encoding='utf-8', errors='replace') as fin, \
             open(cleaned_path, 'w', encoding='utf-8', newline='') as fout:
            buffer = ''
            quote_count = 0
            for raw_line in fin:
                buffer += raw_line
                quote_count += raw_line.count('"')
                # If quotes balanced => assume record complete
                if quote_count % 2 == 0:
                    fout.write(buffer)
                    written += 1
                    buffer = ''
                    quote_count = 0
            # if anything left, write it
            if buffer:
                fout.write(buffer)
                written += 1
        print(f"Cleanup done: wrote {written} logical lines to {cleaned_path}")
        print("Attempting to read cleaned CSV with engine='python'...")
        df = pd.read_csv(cleaned_path, engine='python', on_bad_lines='warn')
        used_path = cleaned_path
        print("Successfully read cleaned CSV.")
    except Exception as e:
        print("Cleanup + read failed:", type(e).__name__, str(e))
        traceback.print_exc()
        raise RuntimeError("Failed to read dataset. Inspect the original CSV manually.") from e

# Final checks & show results
print("\nFinal dataframe shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head(10))

# Make DATA_PATH point to the file that was actually read for later cells
DATA_PATH = used_path
print("\nSet DATA_PATH to:", DATA_PATH)

# Save a small sample for quick inspection
sample_preview_path = '/content/nerfinance_preview.csv'
df.head(200).to_csv(sample_preview_path, index=False)
print("Wrote a preview of the first 200 rows to", sample_preview_path)


Attempting to load dataset from: /content/NERfinance.csv
Trying: default C engine
Strategy failed (default C engine): ParserError Error tokenizing data. C error: EOF inside string starting at row 542
Trying: python engine (more permissive)
Strategy failed (python engine (more permissive)): ParserError unexpected end of data
Trying: python engine, on_bad_lines='warn'
Success with strategy: python engine, on_bad_lines='warn'

Final dataframe shape: (541, 4)
Columns: ['text', 'tokens', 'ner_tags', 'category']


/tmp/ipython-input-1373926143.py:14: ParserWarning: Skipping line 543: unexpected end of data

  return pd.read_csv(path, **kwargs)


,text,tokens,ner_tags,category
0,3x es teh 3k,3x es teh 3k,B-QUANTITY B-ITEM I-ITEM B-PRICE,drink
1,beli 2 nasi goreng @15rb,beli 2 nasi goreng @15rb,O B-QUANTITY B-ITEM I-ITEM O B-PRICE,food
2,"ongkir 12.5rb , total 27rb","ongkir 12.5rb , total 27rb",B-ITEM B-PRICE O B-TOTAL B-PRICE,transport
3,kopi susu 12k aja,kopi susu 12k aja,B-ITEM I-ITEM B-PRICE O,drink
4,bayar kos 1.5jt,bayar kos 1.5jt,O B-ITEM B-PRICE,other
5,2x batagor 8rb,2x batagor 8rb,B-QUANTITY B-ITEM B-PRICE,food
6,"pulsa 50k , kuota 100k","pulsa 50k , kuota 100k",B-ITEM B-PRICE O B-ITEM B-PRICE,shopping
7,es jeruk 5k,es jeruk 5k,B-ITEM I-ITEM B-PRICE,drink
8,traktir temen 75rb,traktir temen 75rb,O B-ITEM B-PRICE,other
9,"bensin 30k , parkir 5k","bensin 30k , parkir 5k",B-ITEM B-PRICE O B-ITEM B-PRICE,transport



Set DATA_PATH to: /content/NERfinance.csv
Wrote a preview of the first 200 rows to /content/nerfinance_preview.csv


In [5]:
#@title Cell 3 — Preprocess, build label map, and create HuggingFace tokenized dataset
import os, json, ast, math
from sklearn.model_selection import train_test_split
import pandas as pd

# Use DATA_PATH set by previous cell (cleaned/original)
try:
    DATA_PATH
except NameError:
    DATA_PATH = '/content/NERfinance.csv'
print("Using DATA_PATH =", DATA_PATH)

# Load CSV (again, safe read in case previous cell didn't persist df)
try:
    df = pd.read_csv(DATA_PATH, engine='python', on_bad_lines='warn')
except Exception as e:
    # fallback: try default pandas
    df = pd.read_csv(DATA_PATH)

print("Loaded dataframe shape:", df.shape)
print("Columns:", df.columns.tolist())

# Heuristics to pick text and label columns
text_candidates = ['text','sentence','utterance','raw','line','input']
label_candidates = ['labels','label','entities','spans','annotation']

text_col = None
labels_col = None
for c in df.columns:
    if c.lower() in text_candidates:
        text_col = c
        break
if text_col is None:
    # fallback to first column
    text_col = df.columns[0]

for c in df.columns:
    if c.lower() in label_candidates:
        labels_col = c
        break
if labels_col is None:
    # fallback: choose second column if exists and not same as text
    if len(df.columns) > 1:
        cand = df.columns[1]
        if cand != text_col:
            labels_col = cand
    else:
        labels_col = None

print("Detected text column:", text_col)
print("Detected labels column:", labels_col)

def parse_label_field(text, label_field):
    """
    Return (tokens:list[str], labels:list[str]) with BIO labels aligned to whitespace tokens.
    Supports:
      - space-separated BIO token labels: "B-ITEM I-ITEM O O"
      - JSON spans: [{'start':10,'end':14,'type':'ITEM'}, ...] OR [{'start_token':0,'end_token':2,'type':'ITEM'}]
      - None/NaN: produce all 'O'
    """
    tokens = str(text).split()
    # If label_field is NaN
    if pd.isna(label_field):
        return tokens, ['O'] * len(tokens)
    # If looks like space-separated BIO labels
    if isinstance(label_field, str) and ('B-' in label_field or 'I-' in label_field) and ' ' in label_field:
        labs = label_field.strip().split()
        # If mismatch lengths, try to align crudely: truncate or pad with O
        if len(labs) != len(tokens):
            if len(labs) > len(tokens):
                labs = labs[:len(tokens)]
            else:
                labs = labs + ['O']*(len(tokens)-len(labs))
        return tokens, labs
    # Otherwise try to parse as JSON spans or Python list
    spans = None
    try:
        if isinstance(label_field, str):
            spans = ast.literal_eval(label_field)
        else:
            spans = label_field
    except Exception:
        # not parseable -> fallback to 'O's
        return tokens, ['O']*len(tokens)

    # If spans is a dict or list of dicts
    if isinstance(spans, dict):
        # maybe single span wrapped in dict
        spans = [spans]

    labels = ['O'] * len(tokens)
    # Precompute token char positions for basic char->token mapping
    token_positions = []
    cursor = 0
    text_str = str(text)
    for tok in tokens:
        # find token starting at or after cursor
        idx = text_str.find(tok, cursor)
        if idx == -1:
            # fallback: approximate positions; assume tokens contiguous
            start = cursor
        else:
            start = idx
        end = start + len(tok)
        token_positions.append((start, end))
        cursor = end

    for sp in spans:
        # support token-index spans
        if isinstance(sp, dict) and 'start_token' in sp and 'end_token' in sp:
            s = int(sp['start_token']); e = int(sp['end_token']); typ = sp.get('type','ENT')
        elif isinstance(sp, dict) and 'start' in sp and 'end' in sp:
            # char offset spans
            char_s = int(sp['start']); char_e = int(sp['end']); typ = sp.get('type','ENT')
            # find first token with end>char_s and last token with start<char_e
            s = next((i for i,(a,b) in enumerate(token_positions) if b > char_s), None)
            e = next((i for i,(a,b) in reversed(list(enumerate(token_positions))) if a < char_e), None)
        else:
            # Unknown span format; skip
            continue
        if s is None or e is None:
            continue
        # clamp indices
        s = max(0, min(s, len(tokens)-1))
        e = max(0, min(e, len(tokens)-1))
        labels[s] = 'B-' + typ
        for j in range(s+1, e+1):
            labels[j] = 'I-' + typ
    return tokens, labels

# Build records list
records = []
for idx, row in df.iterrows():
    t = str(row[text_col])
    lab_field = row[labels_col] if labels_col is not None else None
    toks, labs = parse_label_field(t, lab_field)
    records.append({"id": str(idx), "text": t, "tokens": toks, "labels": labs})

print("Total examples parsed:", len(records))

# Create train/val/test splits (80/10/10)
train, temp = train_test_split(records, test_size=0.2, random_state=42)
val, test = train_test_split(temp, test_size=0.5, random_state=42)
print("Split sizes -> train:", len(train), "val:", len(val), "test:", len(test))

# Collect label set and build maps
unique_labels = sorted({l for ex in records for l in ex['labels']})
if 'O' not in unique_labels:
    unique_labels = ['O'] + unique_labels
label2id = {lab: i for i, lab in enumerate(unique_labels)}
id2label = {i: lab for lab, i in label2id.items()}

# Save processed JSON and label_map
os.makedirs('data/processed', exist_ok=True)
with open('data/processed/train.json', 'w', encoding='utf-8') as f:
    json.dump(train, f, ensure_ascii=False, indent=2)
with open('data/processed/val.json', 'w', encoding='utf-8') as f:
    json.dump(val, f, ensure_ascii=False, indent=2)
with open('data/processed/test.json', 'w', encoding='utf-8') as f:
    json.dump(test, f, ensure_ascii=False, indent=2)

label_map = {"label2id": label2id, "id2label": id2label}
with open('data/label_map.json', 'w', encoding='utf-8') as f:
    json.dump(label_map, f, ensure_ascii=False, indent=2)

print("Saved processed JSON to data/processed/ and label_map to data/label_map.json")
print("Labels ({}): {}".format(len(unique_labels), unique_labels))

# === Build HuggingFace Dataset and tokenize+align labels ===
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer
MODEL_NAME = "indobenchmark/indobert-base-p1"
print("Loading tokenizer:", MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_and_align(examples):
    # examples: dict with "tokens" and "labels" lists
    tokenized = tokenizer(examples["tokens"], is_split_into_words=True, truncation=True)
    aligned_labels = []
    for i, labs in enumerate(examples["labels"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_word_idx = None
        aligned = []
        for word_idx in word_ids:
            if word_idx is None:
                aligned.append(-100)
            elif word_idx != prev_word_idx:
                aligned.append(label2id[labs[word_idx]])
            else:
                # subword continuation: if label is B- -> convert to I-
                lab = labs[word_idx]
                if lab.startswith("B-"):
                    lab = "I-" + lab[2:]
                aligned.append(label2id.get(lab, label2id[labs[word_idx]]))
            prev_word_idx = word_idx
        aligned_labels.append(aligned)
    tokenized["labels"] = aligned_labels
    return tokenized

# Create HF Datasets
ds_train = Dataset.from_list(train)
ds_val = Dataset.from_list(val)
ds_test = Dataset.from_list(test)

# map tokenization
ds_train = ds_train.map(tokenize_and_align, batched=True, remove_columns=['tokens','labels','text','id'])
ds_val = ds_val.map(tokenize_and_align, batched=True, remove_columns=['tokens','labels','text','id'])
ds_test = ds_test.map(tokenize_and_align, batched=True, remove_columns=['tokens','labels','text','id'])

dataset = DatasetDict({"train": ds_train, "validation": ds_val, "test": ds_test})
os.makedirs('data', exist_ok=True)
dataset.save_to_disk('data/hf_dataset')

# Save tokenizer copy in data/
tokenizer.save_pretrained('data/tokenizer_saved')

print("Saved HuggingFace dataset to data/hf_dataset and tokenizer to data/tokenizer_saved")
print("Dataset summary:")
for k in dataset:
    print(k, "->", len(dataset[k]))

# show a sample tokenized example to verify alignment
sample = dataset['train'][0]
print("\nSample tokenized keys:", list(sample.keys()))
print("Input ids length:", len(sample['input_ids']))
print("Labels length:", len(sample['labels']))
# Convert token ids back to tokens for human check
tokens = tokenizer.convert_ids_to_tokens(sample['input_ids'])
print("Tokens (first 60):", tokens[:60])
print("Labels (first 60):", sample['labels'][:60])

# Done
print("\nPreprocessing complete — continue to training cell (Trainer) next.")


Using DATA_PATH = /content/NERfinance.csv
Loaded dataframe shape: (541, 4)
Columns: ['text', 'tokens', 'ner_tags', 'category']
Detected text column: text
Detected labels column: tokens
Total examples parsed: 541
Split sizes -> train: 432 val: 54 test: 55
Saved processed JSON to data/processed/ and label_map to data/label_map.json
Labels (1): ['O']


/tmp/ipython-input-3315698369.py:15: ParserWarning: Skipping line 543: unexpected end of data

  df = pd.read_csv(DATA_PATH, engine='python', on_bad_lines='warn')


Loading tokenizer: indobenchmark/indobert-base-p1


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

...

Map:   0%|          | 0/55 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/432 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/54 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/55 [00:00<?, ? examples/s]

Saved HuggingFace dataset to data/hf_dataset and tokenizer to data/tokenizer_saved
Dataset summary:
train -> 432
validation -> 54
test -> 55

Sample tokenized keys: ['labels', 'input_ids', 'token_type_ids', 'attention_mask']
Input ids length: 6
Labels length: 6
Tokens (first 60): ['[CLS]', 'grab', 'food', '60', '##rb', '[SEP]']
Labels (first 60): [-100, 0, 0, 0, 0, -100]

Preprocessing complete — continue to training cell (Trainer) next.


In [9]:
!pip install evaluate seqeval

In [10]:
!pip install matplotlib-venn

In [11]:
!apt-get -qq install -y libfluidsynth1

E: Package 'libfluidsynth1' has no installation candidate


In [12]:
# https://pypi.python.org/pypi/pydot
!apt-get -qq install -y graphviz && pip install pydot
import pydot

In [13]:
!pip install cartopy
import cartopy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 114.9 MB/s eta 0:00:00


In [16]:
# ===========================
#   CLEAN MINIMAL TRAINING
# ===========================

import numpy as np
import traceback, pprint, evaluate
from transformers import TrainingArguments, Trainer, AutoModelForTokenClassification, DataCollatorForTokenClassification
from datasets import load_from_disk

# Re-load important objects from disk
HF_DATASET_PATH = "data/hf_dataset"
LABEL_MAP_PATH = "data/label_map.json"
MODEL_NAME = "indobenchmark/indobert-base-p1"
OUT_DIR = "models_exported/indobert-expense-ner-silver"

# load dataset
dataset = load_from_disk(HF_DATASET_PATH)

# load label maps
import json
with open(LABEL_MAP_PATH, "r") as f:
    lm = json.load(f)
label2id = lm["label2id"]
id2label = {int(k): v for k, v in lm["id2label"].items()}

# reload tokenizer + model
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# COLLATOR
data_collator = DataCollatorForTokenClassification(tokenizer)

# >>> CREATE training_args HERE (so the variable exists)
training_args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    logging_steps=50,
    save_strategy="no",   # IMPORTANT: avoids eval/save mismatch errors
)

print("TrainingArguments created:", training_args)

# ===== Create Trainer (NO tokenizer kw) =====
try:
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset["train"],
        data_collator=data_collator
    )
    print("Trainer created successfully.")
except Exception as e:
    print("Trainer failed:")
    raise e

# ===== Train =====
try:
    print("Starting training ...")
    trainer.train()
    print("Training complete.")
except Exception as e:
    print("Training error:")
    traceback.print_exc()

# Save model
trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print("Model saved to:", OUT_DIR)

# ===== Manual Evaluation =====
try:
    print("Running predict() on test split ...")
    preds_output = trainer.predict(dataset["test"])
    logits, label_ids = preds_output.predictions, preds_output.label_ids
    preds = np.argmax(logits, axis=2)

    def align_preds(preds_arr, labels_arr):
        pred_list, true_list = [], []
        for p, l in zip(preds_arr, labels_arr):
            p_seq, l_seq = [], []
            for p_i, l_i in zip(p, l):
                if l_i == -100:
                    continue
                p_seq.append(id2label[int(p_i)])
                l_seq.append(id2label[int(l_i)])
            pred_list.append(p_seq)
            true_list.append(l_seq)
        return pred_list, true_list

    pred_list, true_list = align_preds(preds, label_ids)
    metric = evaluate.load("seqeval")
    results = metric.compute(predictions=pred_list, references=true_list)

    print("\nFinal evaluation (seqeval):")
    pprint.pprint(results)

except Exception:
    print("Evaluation failed:")
    traceback.print_exc()

# ===== Demo Prediction =====
try:
    from transformers import pipeline
    nlp = pipeline(
        "token-classification",
        model=OUT_DIR,
        tokenizer=OUT_DIR,
        aggregation_strategy="simple"
    )
    samples = ["grab food 60rb", "beli pulsa 25k", "parkir 10k"]
    for s in samples:
        print(s, "->", nlp(s))
except Exception:
    print("Demo failed:")
    traceback.print_exc()


model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing form the checkpoint. Consider training on your downstream task.


TrainingArguments created: TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.NO,
eval_use_gather_object=False,
f

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


...


Final evaluation (seqeval):
{'overall_accuracy': 1.0,
 'overall_f1': np.float64(0.0),
 'overall_precision': np.float64(0.0),
 'overall_recall': np.float64(0.0)}


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:557: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis, **keepdims_kw)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


grab food 60rb -> []
beli pulsa 25k -> []
parkir 10k -> []


In [17]:
# Diagnostic A: show a tokenized example (input_ids -> tokens) and its labels
from transformers import AutoTokenizer
import json

tokenizer = AutoTokenizer.from_pretrained("data/tokenizer_saved", use_fast=True)  # tokenizer you saved
from datasets import load_from_disk
ds = load_from_disk("data/hf_dataset")

# pick an example index that exists
idx = 0
ex = ds["train"][idx]
print("Keys in example:", ex.keys())
print("Length input_ids:", len(ex["input_ids"]), "labels len:", len(ex["labels"]))
tokens = tokenizer.convert_ids_to_tokens(ex["input_ids"])
print("Tokens:", tokens[:120])
print("Labels (aligned):", ex["labels"][:120])

# Show word_ids mapping for that example to verify alignment
# We need the tokenizer applied to the original tokens to inspect word_ids
original_tokens = tokenizer.convert_tokens_to_string(tokens).split()
print("Preview original (joined) tokens:", original_tokens[:60])
# We can also print counts of -100 vs real labels:
num_masked = sum(1 for l in ex["labels"] if l == -100)
print("Labels -100 count:", num_masked, " / ", len(ex["labels"]))


Keys in example: dict_keys(['labels', 'input_ids', 'token_type_ids', 'attention_mask'])
Length input_ids: 6 labels len: 6
Tokens: ['[CLS]', 'grab', 'food', '60', '##rb', '[SEP]']
Labels (aligned): [-100, 0, 0, 0, 0, -100]
Preview original (joined) tokens: ['[CLS]', 'grab', 'food', '60rb', '[SEP]']
Labels -100 count: 2  /  6


In [18]:
# ===== Fix auto-labeler, rebuild dataset, retokenize, retrain & evaluate =====
# Run this cell (it may take a few minutes). It REPLACES the silver label creation with a safer method.

import re, json, os, pprint, traceback
from datasets import Dataset, DatasetDict, load_from_disk
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification, pipeline
import numpy as np
import evaluate

# paths
PROC_DIR = 'data/processed'
SILVER_DIR = os.path.join(PROC_DIR, 'silver_fixed')
HF_DIR = 'data/hf_dataset'
TOKENIZER_SAVE = 'data/tokenizer_saved'
MODEL_NAME = "indobenchmark/indobert-base-p1"
OUT_DIR = "models_exported/indobert-expense-ner-silver-fixed"

# load the existing processed (original tokens + labels that may be wrong)
def load_json(p):
    with open(p,'r',encoding='utf-8') as f:
        return json.load(f)

train = load_json(os.path.join(PROC_DIR,'train.json'))
val   = load_json(os.path.join(PROC_DIR,'val.json'))
test  = load_json(os.path.join(PROC_DIR,'test.json'))
print("Loaded processed counts:", len(train), len(val), len(test))

# Print the first example (for inspection)
print("\n=== Example BEFORE fix (first train sample) ===")
print("text:", train[0]['text'])
print("tokens:", train[0]['tokens'])
print("labels:", train[0]['labels'])
print("----")

# Safer auto-labeler:
# Strategy:
# 1) Use regex to find price/qty spans in the raw text (char offsets).
# 2) Build token char spans robustly by scanning the text using re.finditer(r'\S+') to find each whitespace-separated token and its offsets.
# 3) For each regex match, mark overlapping tokens only (so we won't accidentally mark preceding tokens).
price_patterns = [
    r'\b\d{1,3}(?:[.,]\d{3})+(?:\s*(?:k|K|rb|RB|r|R|rupiah|Rp|rp|jt|JT))?\b',
    r'\b\d+(?:[.,]\d+)?\s*(?:k|K|rb|RB|r|R|rupiah|Rp|rp|jt|JT)\b',
    r'\brp\s*\d+\b',
    r'\b\d+\s*(?:k|K|rb|jt|JT)\b',
]
qty_patterns = [r'\b\d+\s*[xX]\b', r'\b\d+\s*(?:pcs|pc|pcs\.)\b']

price_re = re.compile("|".join(price_patterns), flags=re.IGNORECASE)
qty_re   = re.compile("|".join(qty_patterns), flags=re.IGNORECASE)

def token_spans_from_text(text):
    """Return list of (token, start, end) using regex split which is robust."""
    spans = []
    for m in re.finditer(r'\S+', text):
        spans.append((m.group(0), m.start(), m.end()))
    return spans

def safe_auto_label(text, orig_labels=None):
    """
    Given raw text and optional orig_labels (aligned to whitespace tokens),
    produce (tokens, labels) where only tokens overlapping a regex match get PRICE/QTY.
    """
    token_spans = token_spans_from_text(text)  # list of (tok, s, e)
    tokens = [t for t,_,_ in token_spans]
    labels = ['O'] * len(tokens)

    # mark price spans
    for m in price_re.finditer(text):
        s,e = m.start(), m.end()
        for i,(tok,ts,te) in enumerate(token_spans):
            # overlap condition
            if te <= s or ts >= e:
                continue
            # mark B/I appropriately
            if labels[i] == 'O':
                labels[i] = 'B-PRICE'
            else:
                labels[i] = 'I-PRICE'

    # mark qty spans (but do not overwrite PRICE)
    for m in qty_re.finditer(text):
        s,e = m.start(), m.end()
        for i,(tok,ts,te) in enumerate(token_spans):
            if te <= s or ts >= e:
                continue
            if labels[i] == 'O':
                labels[i] = 'B-QTY'
            else:
                # if it's already PRICE, keep PRICE
                pass

    # if orig_labels exist and contain non-O, preserve them (merge)
    if orig_labels:
        # if lengths mismatch, attempt alignment by assuming orig_labels map to whitespace tokens
        for i in range(min(len(labels), len(orig_labels))):
            if orig_labels[i] != 'O':
                labels[i] = orig_labels[i]

    return tokens, labels

# Create fixed silver splits
def fix_split(split):
    fixed=[]
    for ex in split:
        text = ex.get('text') or ex.get('raw') or ''
        orig_labels = ex.get('labels', None)
        toks, labs = safe_auto_label(text, orig_labels=orig_labels)
        fixed.append({"id": ex.get('id',''), "text": text, "tokens": toks, "labels": labs})
    return fixed

train_f = fix_split(train)
val_f   = fix_split(val)
test_f  = fix_split(test)

# Save fixed silver processed
os.makedirs(SILVER_DIR, exist_ok=True)
with open(os.path.join(SILVER_DIR,'train.json'),'w',encoding='utf-8') as f: json.dump(train_f,f,ensure_ascii=False,indent=2)
with open(os.path.join(SILVER_DIR,'val.json'),'w',encoding='utf-8') as f: json.dump(val_f,f,ensure_ascii=False,indent=2)
with open(os.path.join(SILVER_DIR,'test.json'),'w',encoding='utf-8') as f: json.dump(test_f,f,ensure_ascii=False,indent=2)
print("\nSaved fixed silver processed to:", SILVER_DIR)

# Build label map
labels_set = {'O'}
for s in (train_f, val_f, test_f):
    for ex in s:
        labels_set.update([l for l in ex['labels'] if l!='O'])
labels_list = sorted(labels_set)
label2id = {lab:i for i,lab in enumerate(labels_list)}
id2label = {i:lab for lab,i in label2id.items()}
with open('data/label_map.json','w',encoding='utf-8') as f: json.dump({"label2id":label2id,"id2label":id2label}, f, ensure_ascii=False, indent=2)
pprint.pprint({"label2id":label2id})

# Quick sanity print of a few fixed samples
print("\n=== Some fixed samples (first 6 train) ===")
for ex in train_f[:6]:
    print(ex['text'])
    print("tokens:", ex['tokens'])
    print("labels:", ex['labels'])
    print("---")

# ---- Tokenize & align to HF dataset ----
print("\nTokenizing & aligning (this may take a bit)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_and_align_fixed(examples):
    tokenized = tokenizer(examples["tokens"], is_split_into_words=True, truncation=True)
    aligned = []
    for i,labs in enumerate(examples["labels"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev = None
        al = []
        for wid in word_ids:
            if wid is None:
                al.append(-100)
            elif wid != prev:
                al.append(label2id[labs[wid]])
            else:
                lab = labs[wid]
                if lab.startswith("B-"):
                    lab = "I-"+lab[2:]
                al.append(label2id.get(lab, label2id[labs[wid]]))
            prev = wid
        aligned.append(al)
    tokenized["labels"] = aligned
    return tokenized

ds_train = Dataset.from_list(train_f)
ds_val   = Dataset.from_list(val_f)
ds_test  = Dataset.from_list(test_f)

ds_train = ds_train.map(tokenize_and_align_fixed, batched=True, remove_columns=['tokens','labels','text','id'])
ds_val   = ds_val.map(tokenize_and_align_fixed, batched=True, remove_columns=['tokens','labels','text','id'])
ds_test  = ds_test.map(tokenize_and_align_fixed, batched=True, remove_columns=['tokens','labels','text','id'])

dataset = DatasetDict({"train": ds_train, "validation": ds_val, "test": ds_test})
dataset.save_to_disk(HF_DIR)
tokenizer.save_pretrained(TOKENIZER_SAVE)
print("Saved HF dataset & tokenizer. Sizes:", {k: len(dataset[k]) for k in dataset})

# ---- Train (minimal args to avoid earlier constructor problems) ----
print("\nStarting training with corrected labels...")
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, num_labels=len(label2id), id2label=id2label, label2id=label2id)
data_collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    logging_steps=50,
    save_strategy="no",
    fp16=False
)

trainer = Trainer(model=model, args=training_args, train_dataset=dataset["train"], data_collator=data_collator)

trainer.train()
trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print("Training finished and model saved to:", OUT_DIR)

# ---- Manual prediction & evaluation ----
print("\nRunning predict() on test split...")
preds_output = trainer.predict(dataset["test"])
logits, label_ids = preds_output.predictions, preds_output.label_ids
preds = np.argmax(logits, axis=2)

def align_preds(preds_arr, label_ids_arr):
    pred_list, true_list = [], []
    for p, l in zip(preds_arr, label_ids_arr):
        p_seq, l_seq = [], []
        for p_i, l_i in zip(p, l):
            if l_i == -100: continue
            p_seq.append(id2label.get(int(p_i), "O"))
            l_seq.append(id2label.get(int(l_i), "O"))
        pred_list.append(p_seq)
        true_list.append(l_seq)
    return pred_list, true_list

pred_list, true_list = align_preds(preds, label_ids)
metric = evaluate.load("seqeval")
results = metric.compute(predictions=pred_list, references=true_list)
print("\nSeqeval results after fixing labels:")
pprint.pprint(results)

print("\nDemo (pipeline) outputs:")
nlp = pipeline("token-classification", model=OUT_DIR, tokenizer=OUT_DIR, aggregation_strategy="simple")
examples = ["grab food 60rb", "beli pulsa 25k", "parkir 10k", "bayar kontrakan 3.8jt"]
for ex in examples:
    print("-", ex, "->", nlp(ex))

print("\nDone. If results still show zero entities, paste the seqeval + a couple of training samples and I'll iterate.")


Loaded processed counts: 432 54 55

=== Example BEFORE fix (first train sample) ===
text: grab food 60rb
tokens: ['grab', 'food', '60rb']
labels: ['O', 'O', 'O']
----

Saved fixed silver processed to: data/processed/silver_fixed
{'label2id': {'B-PRICE': 0, 'B-QTY': 1, 'O': 2}}

=== Some fixed samples (first 6 train) ===
grab food 60rb
tokens: ['grab', 'food', '60rb']
labels: ['O', 'O', 'B-PRICE']
---
beli token 450k
tokens: ['beli', 'token', '450k']
labels: ['O', 'O', 'B-PRICE']
---
grab ke mall 85k
tokens: ['grab', 'ke', 'mall', '85k']
labels: ['O', 'O', 'O', 'B-PRICE']
---
3x kopi hitam 25k
tokens: ['3x', 'kopi', 'hitam', '25k']
labels: ['B-QTY', 'O', 'O', 'B-PRICE']
---
ongkir shopee 28k
tokens: ['ongkir', 'shopee', '28k']
labels: ['O', 'O', 'B-PRICE']
---
parkir 22k
tokens: ['parkir', '22k']
labels: ['O', 'B-PRICE']
---

Tokenizing & aligning (this may take a bit)...


Map:   0%|          | 0/432 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Map:   0%|          | 0/54 [00:00<?, ? examples/s]

Map:   0%|          | 0/55 [00:00<?, ? examples/s]

...

Training finished and model saved to: models_exported/indobert-expense-ner-silver-fixed

Running predict() on test split...



Seqeval results after fixing labels:
{'PRICE': {'f1': np.float64(1.0),
           'number': np.int64(118),
           'precision': np.float64(1.0),
           'recall': np.float64(1.0)},
 'QTY': {'f1': np.float64(1.0),
         'number': np.int64(10),
         'precision': np.float64(1.0),
         'recall': np.float64(1.0)},
 'overall_accuracy': 1.0,
 'overall_f1': np.float64(1.0),
 'overall_precision': np.float64(1.0),
 'overall_recall': np.float64(1.0)}

Demo (pipeline) outputs:


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


- grab food 60rb -> [{'entity_group': 'PRICE', 'score': np.float32(0.9999982), 'word': '60', 'start': 10, 'end': 12}, {'entity_group': 'PRICE', 'score': np.float32(0.99999464), 'word': '##rb', 'start': 12, 'end': 14}]
- beli pulsa 25k -> [{'entity_group': 'PRICE', 'score': np.float32(0.99999785), 'word': '25', 'start': 11, 'end': 13}, {'entity_group': 'PRICE', 'score': np.float32(0.99999285), 'word': '##k', 'start': 13, 'end': 14}]
- parkir 10k -> [{'entity_group': 'PRICE', 'score': np.float32(0.9999968), 'word': '10', 'start': 7, 'end': 9}, {'entity_group': 'PRICE', 'score': np.float32(0.9999881), 'word': '##k', 'start': 9, 'end': 10}]
- bayar kontrakan 3.8jt -> [{'entity_group': 'PRICE', 'score': np.float32(0.9999931), 'word': '3', 'start': 16, 'end': 17}, {'entity_group': 'PRICE', 'score': np.float32(0.99997306), 'word': '.', 'start': 17, 'end': 18}, {'entity_group': 'PRICE', 'score': np.float32(0.9999753), 'word': '8', 'start': 18, 'end': 19}, {'entity_group': 'PRICE', 'score': np.

In [19]:
# Diagnostic B: count label ids in dataset (how many token labels per id)
from collections import Counter
ds = load_from_disk("data/hf_dataset")
c = Counter()
for split in ["train","validation","test"]:
    for ex in ds[split]:
        for l in ex["labels"]:
            c[l] += 1
print("Label id counts (including -100):")
for k,v in sorted(c.items()):
    print(k, ":", v)
# Also show mapping label id -> label name (from data/label_map.json)
import json
lm = json.load(open("data/label_map.json","r",encoding="utf-8"))
print("label2id:", lm.get("label2id"))
print("id2label:", lm.get("id2label"))


Label id counts (including -100):
-100 : 1082
0 : 1120
1 : 170
2 : 1365
label2id: {'B-PRICE': 0, 'B-QTY': 1, 'O': 2}
id2label: {'0': 'B-PRICE', '1': 'B-QTY', '2': 'O'}


In [21]:
# Corrected Diagnostic C: safe batch -> tensor conversion, forward pass, and debug prints
import torch, numpy as np, traceback
from transformers import AutoModelForTokenClassification
from datasets import load_from_disk
from torch.utils.data import DataLoader

ds = load_from_disk("data/hf_dataset")

# load a small sample
sample_n = min(8, len(ds["train"]))
subset = ds["train"].select(list(range(sample_n)))

# create DataLoader (huggingface Arrow format returns lists/arrays)
def collate_fn(batch):
    # batch is a list of dicts with same keys
    collated = {}
    for k in batch[0].keys():
        vals = [b[k] for b in batch]
        collated[k] = vals
    return collated

loader = DataLoader(subset, batch_size=2, collate_fn=collate_fn)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForTokenClassification.from_pretrained("models_exported/indobert-expense-ner-silver")
model.to(device)
model.eval()

batch = next(iter(loader))

def to_tensor(v):
    # safe conversion to tensor on device
    if torch.is_tensor(v):
        return v.to(device)
    if isinstance(v, np.ndarray):
        return torch.from_numpy(v).to(device)
    if isinstance(v, list):
        try:
            return torch.tensor(v).to(device)
        except Exception:
            # try to convert nested lists to numpy first
            try:
                return torch.from_numpy(np.array(v)).to(device)
            except Exception:
                raise
    # fallback: try converting scalar
    try:
        return torch.tensor(v).to(device)
    except Exception:
        raise TypeError(f"Cannot convert batch field of type {type(v)} to tensor")

# convert batch safely
batch_t = {}
for k,v in batch.items():
    try:
        batch_t[k] = to_tensor(v)
    except Exception as e:
        print("Skipping key", k, "due to conversion error:", e)

print("Converted keys:", list(batch_t.keys()))
# Ensure required keys exist
required = ["input_ids","attention_mask","labels"]
for r in required:
    if r not in batch_t:
        print("Warning: key missing in batch_t:", r)

# Forward pass (wrapped)
try:
    with torch.no_grad():
        outputs = model(**batch_t)
    print("Forward OK. Loss:", getattr(outputs, "loss", None))
    if hasattr(outputs, "logits"):
        print("Logits shape:", outputs.logits.shape)
    # Inspect a few labels & preds
    logits = outputs.logits.detach().cpu().numpy()
    labels = batch_t.get("labels").detach().cpu().numpy() if "labels" in batch_t else None
    if labels is not None:
        print("Labels sample (first example):", labels[0][:50])
        preds = logits.argmax(axis=-1)
        print("Preds sample (first example):", preds[0][:50])
except Exception:
    print("Forward pass failed; traceback:")
    traceback.print_exc()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Converted keys: ['labels', 'input_ids', 'token_type_ids', 'attention_mask']
Forward pass failed; traceback:


Traceback (most recent call last):
  File "/tmp/ipython-input-3316671136.py", line 70, in <cell line: 0>
    outputs = model(**batch_t)
              ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1773, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1784, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py", line 768, in wrapper
    output = func(self, *args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/bert/modeling_bert.py", line 1364, in forward
    loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr

In [22]:
import json
from transformers import AutoModelForTokenClassification

LABEL_MAP_PATH = "data/label_map.json"
MODEL_SAVED = "models_exported/indobert-expense-ner-silver"  # change if different

lm = json.load(open(LABEL_MAP_PATH, 'r', encoding='utf-8'))
print("label_map:", lm)
label2id = lm['label2id']
id2label = lm['id2label']
print("num labels (dataset):", len(label2id))

# load saved model config (no weights needed) to inspect num_labels
try:
    m = AutoModelForTokenClassification.from_pretrained(MODEL_SAVED, local_files_only=True)
    print("Saved model config num_labels:", m.config.num_labels)
except Exception as e:
    print("Could not load saved model from", MODEL_SAVED, ":", e)
    # try reading config.json manually
    import os, json
    cfg_path = os.path.join(MODEL_SAVED, "config.json")
    if os.path.exists(cfg_path):
        cfg = json.load(open(cfg_path,'r',encoding='utf-8'))
        print("config.json num_labels:", cfg.get("num_labels"))
    else:
        print("No config.json at", cfg_path)


label_map: {'label2id': {'B-PRICE': 0, 'B-QTY': 1, 'O': 2}, 'id2label': {'0': 'B-PRICE', '1': 'B-QTY', '2': 'O'}}
num labels (dataset): 3


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Saved model config num_labels: 1


In [23]:
from transformers import AutoModelForTokenClassification

MODEL_NAME = "indobenchmark/indobert-base-p1"
# dataset label map
import json
lm = json.load(open("data/label_map.json","r",encoding="utf-8"))
label2id = lm['label2id']
id2label = {int(k):v for k,v in lm['id2label'].items()}
num_labels = len(label2id)
print("Recreating model with num_labels =", num_labels, "and id2label:", id2label)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# save/overwrite the model folder so later loading picks up correct head shape (optional)
OUT = "models_exported/indobert-expense-ner-silver-fixed-head"
model.save_pretrained(OUT)
print("Saved new model with correct head to", OUT)


Recreating model with num_labels = 3 and id2label: {0: 'B-PRICE', 1: 'B-QTY', 2: 'O'}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing form the checkpoint. Consider training on your downstream task.


Saved new model with correct head to models_exported/indobert-expense-ner-silver-fixed-head


In [24]:
# Run a forward pass to ensure no IndexError
import torch, numpy as np
from datasets import load_from_disk
from torch.utils.data import DataLoader

ds = load_from_disk("data/hf_dataset")
sample = ds["train"].select(range(min(8, len(ds["train"]))))
def collate(batch):
    coll = {}
    for k in batch[0].keys():
        coll[k] = [b[k] for b in batch]
    return coll

loader = DataLoader(sample, batch_size=2, collate_fn=collate)
batch = next(iter(loader))

# convert lists/arrays -> tensors safely
def to_tensor(v):
    import numpy as np, torch
    if torch.is_tensor(v): return v
    if isinstance(v, np.ndarray): return torch.from_numpy(v)
    try:
        return torch.tensor(v)
    except:
        return torch.tensor(np.array(v))

batch_t = {k: to_tensor(v).to('cuda' if torch.cuda.is_available() else 'cpu') for k,v in batch.items() if k in ("input_ids","attention_mask","labels")}
model.to('cuda' if torch.cuda.is_available() else 'cpu')
model.eval()
with torch.no_grad():
    out = model(**batch_t)
print("Forward OK — loss:", getattr(out, "loss", None), "logits shape:", out.logits.shape)


Forward OK — loss: tensor(1.2487) logits shape: torch.Size([2, 6, 3])


In [25]:
# Minimal training using the corrected model (this will continue training the head)
from transformers import Trainer, TrainingArguments, DataCollatorForTokenClassification, AutoTokenizer
from datasets import load_from_disk
import evaluate, pprint

tokenizer = AutoTokenizer.from_pretrained("data/tokenizer_saved", use_fast=True)
dataset = load_from_disk("data/hf_dataset")
data_collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir="models_exported/indobert-expense-ner-silver-final",
    per_device_train_batch_size=8,
    num_train_epochs=3,
    logging_steps=50,
    save_strategy="no",
    fp16=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    data_collator=data_collator
)

trainer.train()
trainer.save_model(training_args.output_dir)
tokenizer.save_pretrained(training_args.output_dir)
print("Training complete and model saved to", training_args.output_dir)

# Manual eval
preds_output = trainer.predict(dataset["test"])
preds = np.argmax(preds_output.predictions, axis=2)
label_ids = preds_output.label_ids
def align(preds, labels):
    p_list, t_list = [], []
    for p,l in zip(preds, labels):
        p_seq, l_seq = [], []
        for p_i,l_i in zip(p,l):
            if l_i == -100: continue
            p_seq.append(id2label.get(int(p_i),"O"))
            l_seq.append(id2label.get(int(l_i),"O"))
        p_list.append(p_seq); t_list.append(l_seq)
    return p_list, t_list

pred_list, true_list = align(preds, label_ids)
metric = evaluate.load("seqeval")
res = metric.compute(predictions=pred_list, references=true_list)
pprint.pprint(res)


Step,Training Loss
50,0.054300
100,0.000000
150,0.000000


Training complete and model saved to models_exported/indobert-expense-ner-silver-final


{'PRICE': {'f1': np.float64(1.0),
           'number': np.int64(118),
           'precision': np.float64(1.0),
           'recall': np.float64(1.0)},
 'QTY': {'f1': np.float64(1.0),
         'number': np.int64(10),
         'precision': np.float64(1.0),
         'recall': np.float64(1.0)},
 'overall_accuracy': 1.0,
 'overall_f1': np.float64(1.0),
 'overall_precision': np.float64(1.0),
 'overall_recall': np.float64(1.0)}


In [26]:
from transformers import pipeline
ner = pipeline("token-classification",
               model="models_exported/indobert-expense-ner-silver-final",
               tokenizer="models_exported/indobert-expense-ner-silver-final",
               aggregation_strategy="simple")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [27]:
from transformers import pipeline
model_dir = "models_exported/indobert-expense-ner-silver-final"
nlp = pipeline("token-classification", model=model_dir, tokenizer=model_dir, aggregation_strategy="simple")

examples = [
    "grab food 60rb",
    "beli pulsa 25k",
    "parkir 10k",
    "bayar kontrakan 3.8jt",
    "3x kopi hitam 25k",
    "Total 180rb buat jajan",
    "ongkir shopee 28k",
    "Sate 5 tusuk Rp50.000",
    "Bayar 1.2jt sewa kos",
    "2 nasi goreng @ 15k"
]
for ex in examples:
    print(ex, "->", nlp(ex))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


grab food 60rb -> [{'entity_group': 'PRICE', 'score': np.float32(0.99999547), 'word': '60', 'start': 10, 'end': 12}, {'entity_group': 'PRICE', 'score': np.float32(0.99999666), 'word': '##rb', 'start': 12, 'end': 14}]
beli pulsa 25k -> [{'entity_group': 'PRICE', 'score': np.float32(0.9999976), 'word': '25', 'start': 11, 'end': 13}, {'entity_group': 'PRICE', 'score': np.float32(0.9999976), 'word': '##k', 'start': 13, 'end': 14}]
parkir 10k -> [{'entity_group': 'PRICE', 'score': np.float32(0.99999785), 'word': '10', 'start': 7, 'end': 9}, {'entity_group': 'PRICE', 'score': np.float32(0.9999962), 'word': '##k', 'start': 9, 'end': 10}]
bayar kontrakan 3.8jt -> [{'entity_group': 'PRICE', 'score': np.float32(0.9999945), 'word': '3', 'start': 16, 'end': 17}, {'entity_group': 'PRICE', 'score': np.float32(0.99998665), 'word': '.', 'start': 17, 'end': 18}, {'entity_group': 'PRICE', 'score': np.float32(0.9999956), 'word': '8', 'start': 18, 'end': 19}, {'entity_group': 'PRICE', 'score': np.float32(

In [29]:
import os

print("1) Does /mnt/data exist? ->", os.path.exists("/mnt/data"))
print("2) Root directory listing:")
print(os.listdir("/"))

print("\n3) Contents of current working directory:")
print(os.listdir("."))

print("\n4) Contents of models_exported (if exists):")
print(os.listdir("models_exported") if os.path.exists("models_exported") else "models_exported folder NOT found")

print("\n5) Check specific path:")
print("models_exported/indobert-expense-ner-silver-final exists? ->",
      os.path.exists("models_exported/indobert-expense-ner-silver-final"))


1) Does /mnt/data exist? -> False
2) Root directory listing:
['srv', 'media', 'usr', 'lib64', 'lib', 'bin', 'dev', 'run', 'opt', 'sbin', 'etc', 'root', 'libx32', 'boot', 'sys', 'tmp', 'mnt', 'home', 'proc', 'var', 'lib32', 'content', 'kaggle', '.dockerenv', 'tools', 'datalab', 'python-apt', 'python-apt.tar.xz', 'NGC-DL-CONTAINER-LICENSE', 'cuda-keyring_1.1-1_all.deb']

3) Contents of current working directory:
['.config', 'nerfinance_preview.csv', 'data', 'NERfinance.csv', 'models_exported', 'drive', 'sample_data']

4) Contents of models_exported (if exists):
['indobert-expense-ner-silver-final', 'indobert-expense-ner-silver-fixed-head', 'indobert-expense-ner-silver-fixed', 'indobert-expense-ner-silver']

5) Check specific path:
models_exported/indobert-expense-ner-silver-final exists? -> True


In [30]:
%%bash
MODEL_DIR="models_exported/indobert-expense-ner-silver-final"
OUT_ZIP="indobert-expense-ner-model.zip"

rm -f "$OUT_ZIP"
zip -r "$OUT_ZIP" "$MODEL_DIR"

echo "Model zipped at $OUT_ZIP"


  adding: models_exported/indobert-expense-ner-silver-final/ (stored 0%)
  adding: models_exported/indobert-expense-ner-silver-final/special_tokens_map.json (deflated 80%)
  adding: models_exported/indobert-expense-ner-silver-final/tokenizer.json (deflated 71%)
  adding: models_exported/indobert-expense-ner-silver-final/vocab.txt (deflated 53%)
  adding: models_exported/indobert-expense-ner-silver-final/model.safetensors (deflated 7%)
  adding: models_exported/indobert-expense-ner-silver-final/tokenizer_config.json (deflated 74%)
  adding: models_exported/indobert-expense-ner-silver-final/training_args.bin (deflated 53%)
  adding: models_exported/indobert-expense-ner-silver-final/config.json (deflated 53%)
Model zipped at indobert-expense-ner-model.zip
